# Pricing with a knowledge base: retrieval-grounded recommendations

Stage 3 of the pricing agent. In stage 1 the agent priced a product from five numbers and its own judgement, so every market fact in its answers (what similar products sell for, how shoppers react to price) came from the model's memory. Stage 2 showed how to retrieve from a document. Here the pricing agent gets a **knowledge base of the company's own pricing history** to retrieve from, and must cite every fact it uses.

**The knowledge base** holds five kinds of records: elasticity benchmarks, historical margins, past pricing decisions with their outcomes, competitor trends and pricing guidelines. All of it is fictional Indian retail data in rupees (₹).

**The questions:**
1. How should records be retrieved so that the right evidence reaches the model?
2. Does retrieval reduce made-up market facts, compared with the same agent without it?
3. Does the agent use the evidence: does elasticity move the price in the right direction?
4. What does the agent do when the knowledge base knows nothing about a category, and what changes when new data arrives?

Margin always means gross margin on the selling price, `(price − cost) / price`, as in stage 1, so the minimum price for a target margin is `cost / (1 − target margin)`.

## Setup

Install the dependencies from the repository root (`pip install -r requirements.txt`) and put your Groq API key in a `.env` file at the repository root (copy `.env.example`). `pricing_knowledge_rag` loads the key when it is imported. The embedding model runs locally and needs no key.

In [1]:
import pandas as pd

from pricing_knowledge_rag import (
    EXAMPLE_PRODUCT, MODEL, NEW_CATEGORY_PATH, RAGPricingAgent, add_documents, build_store, compare,
    create_documents, explore, load_knowledge, make_embeddings,
)

knowledge = load_knowledge()
print("Model:", MODEL)
for kind, records in knowledge.items():
    print(f"{kind}: {len(records)}")

Model: openai/gpt-oss-120b
elasticity_benchmarks: 6
historical_margins: 6
pricing_decisions: 26
competitor_trends: 6
pricing_guidelines: 8


## 1. The knowledge base

[`knowledge_base.json`](knowledge_base.json) covers six categories: Electronics, Footwear, Apparel, Luxury Goods, Groceries, and Home & Kitchen.

| Kind | Example |
|---|---|
| Elasticity benchmark | Footwear: High. A 5% price rise cuts branded-sneaker sales by 7-10%. |
| Historical margins | Footwear averages 36%; branded sneakers above a 40% margin sold less than 60% of stock at full price. |
| Past pricing decision | Puma Smash at ₹2,399, 4% below the competitor, sold 92% of its stock at full price. |
| Competitor trend | Entry-level branded sneakers cluster in the ₹2,199-₹2,999 band. |
| Guideline | High elasticity: change prices in steps of 3-5% and stay within 3% of the main competitor. |

The past decisions include failures on purpose, such as running shoes priced 14% above a competitor, so the agent can learn what to avoid as well as what worked. Each decision's margin is calculated from its prices when the file is loaded, so it always matches them.

In [2]:
pd.DataFrame(knowledge["pricing_decisions"])[
    ["product", "category", "cost", "recommended_price", "margin", "competitor_price", "elasticity", "outcome"]]

,product,category,cost,recommended_price,margin,competitor_price,elasticity,outcome
0,Samsung Galaxy Smartphone,Electronics,45000,49900,9.8%,52900,High,Successful - gained 15% market share
1,boAt Airdopes Wireless Earbuds,Electronics,1150,1499,23.3%,1599,High,Successful - became the best-selling audio pro...
2,Sony Noise-Cancelling Headphones,Electronics,21500,26990,20.3%,26490,High,Unsuccessful - sales 20% below plan as shopper...
3,HP 15-inch Laptop,Electronics,48000,54990,12.7%,55990,High,Successful - 25% more units than the previous ...
4,LG 43-inch Smart TV,Electronics,27000,29990,10.0%,31490,High,Successful - festive-season volume up 40%
5,25W USB-C Fast Charger,Electronics,380,799,52.4%,849,Medium,Successful - high attach rate with phone sales
6,Puma Smash Sneakers,Footwear,1650,2399,31.2%,2499,High,Successful - sold 92% of stock at full price i...
7,Puma Softride Running Shoes,Footwear,1950,2999,35.0%,2899,High,Unsuccessful - units fell 12% in 4 weeks; pric...
8,Nike Revolution Running Shoes,Footwear,1900,3299,42.4%,2899,High,Unsuccessful - units 28% below plan; marked do...
9,Skechers Go Walk Slip-ons,Footwear,1500,2199,31.8%,2299,High,Successful - 18% more units than the previous ...


## 2. Documents and the vector store

Each record becomes **one document**, so no chunking is needed: the unit we want back is one whole benchmark, decision or guideline, and every record is already short. Each document carries metadata:

- `doc_type`: `elasticity_benchmark`, `historical_margin`, `past_decision`, `competitor_trend` or `guideline`
- `category`: the record's category (`All` for guidelines)
- `applies_to`: for guidelines, the elasticity or situation the rule is for
- `doc_id`: a stable id built from the content, so adding a record again replaces it

The documents are embedded locally with `all-MiniLM-L6-v2` into an in-memory Chroma collection.

In [3]:
documents = create_documents(knowledge)
print(f"{len(documents)} documents\n")
example = next(doc for doc in documents if doc.metadata["doc_type"] == "past_decision")
print(example.page_content)
print(example.metadata)

store = build_store(make_embeddings(), knowledge)
print(f"\nIndexed {store._collection.count()} documents")

52 documents

Past pricing decision: Samsung Galaxy Smartphone (Electronics, High elasticity). Cost ₹45,000, price ₹49,900 (9.8% margin), competitor price ₹52,900. Outcome: Successful - gained 15% market share. Reasoning: Aggressive pricing in high elasticity category drove volume.
{'doc_type': 'past_decision', 'category': 'Electronics', 'doc_id': 'decision:Samsung Galaxy Smartphone'}



Indexed 52 documents


## 3. Why one similarity search is not enough

The obvious retriever embeds the product description and returns the `k` most similar documents. `explore()` shows what that returns, with each match's distance (lower is closer). Three probes: a strategy question, a category question, and a product with and without a metadata filter.

In [4]:
explore(store, "high elasticity pricing strategy")
print("=" * 70 + "\n")
explore(store, "footwear sneakers margin")
print("=" * 70 + "\n")
explore(store, "Coach Leather Handbag (Luxury Goods), Low price elasticity", k=5)
print("=" * 70 + "\n")
_ = explore(store, "Coach Leather Handbag (Luxury Goods), Low price elasticity", k=5,
            filter={"category": "Luxury Goods"})

Query: 'high elasticity pricing strategy'
1. [guideline | All]  distance 0.68
   Pricing guideline - High elasticity: small price changes (applies to High elasticity): Change prices in small steps of no more than 3-5% at a time, and stay within 3% of the main competitor. Prefer pricing slightly below the competitor when the margin allows. Why: Price-sensitive shoppers switch quickly after a large increase, and competitors match large cuts within days.

2. [elasticity_benchmark | Electronics]  distance 0.79
   Elasticity benchmark for Electronics: High price elasticity (coefficient -1.5 to -2.5). A 5% price rise typically cuts unit sales by 8-12%. Shoppers compare prices online before buying, and festive sales (October-November) bring about 35% of yearly volume.

3. [guideline | All]  distance 0.80
   Pricing guideline - Low elasticity: protect price and brand (applies to Low elasticity): Price at or up to 10% above the competitor when the brand justifies it, and avoid discounts above 1

**What the exploration shows** (retrieval involves no model, so these results are the same on every run)

- **Embeddings barely tell "high" from "low".** For "high elasticity pricing strategy", the third result is the *Low* elasticity guideline, at a distance (0.80) almost equal to the second result's (0.79). The two guidelines share most of their words, so their embeddings sit close together, and a context built this way could hand the model the opposite rule.
- **Specific words work well.** "footwear sneakers margin" puts the footwear margin record first (0.43), followed by two more footwear records.
- **Unfiltered searches drift across categories.** For the Coach handbag, results 3 and 4 are the footwear benchmark and an apparel shirt. With `filter={"category": "Luxury Goods"}`, all five results are luxury records, including the competitor trend and the margins that the unfiltered search missed.

This is why the agent filters every search by category and document type, and picks the elasticity guideline by an exact metadata match instead of by similarity.

## 4. The agent

`RAGPricingAgent` runs **one small search per kind of evidence**, each filtered by metadata, so the context always has every kind of evidence and nothing from other categories:

- the category's elasticity benchmark, historical margins and competitor trend (one of each)
- the 4 past decisions in the category most similar to the product
- the guideline for the product's elasticity (given, or else taken from the category benchmark), picked by exact metadata match instead of by similarity
- the 2 general guidelines most similar to the product

If the category is not in the knowledge base at all, the agent fetches the guideline for categories with no history and tells the model there is no category data.

The documents are numbered in the prompt, and the system prompt tells the model to cite them like `[3]` for every market fact, to say so when the context doesn't cover something, and to learn from past outcomes. The answer has four fixed sections: margin floor, evidence, recommendation and risks.

**The baseline** is the same model with the same rules and answer sections, but no knowledge base, like stage 1. Only retrieval differs, so the comparison shows what retrieval adds.

The reference product is the one from stage 1's brief: Puma sneakers, cost ₹1,800, 30% target margin, High elasticity, no competitor price. The margin floor is ₹1,800 / 0.70 = ₹2,571.43.

In [5]:
agent = RAGPricingAgent(store)
print(compare(agent, **EXAMPLE_PRODUCT))

RAG AGENT (10 documents retrieved, 5 cited)
----------------------------------------------------------------------
**1. Margin floor**  
Target margin = 30 % → Minimum price = Cost ÷ (1 – target) = 1,800 ÷ 0.70 = **₹2,571.43**.  
Round up to the next acceptable price point → **₹2,599** (ends in 99)【9】.  

**2. Evidence**  
- High elasticity (‑1.4 to ‑2.0) means a 5 % rise cuts units 7‑10 %【1】 → keep price changes ≤3‑5 %【8】.  
- Branded sneaker market clusters at ₹2,199‑₹2,999【3】; a price in the middle aligns with shopper expectations.  
- Past Puma Smash (cost ₹1,650, price ₹2,399, 31.2 % margin) sold 92 % of stock when priced ~4 % below the competitor【4】 → similar margin works.  
- Guideline for high‑elasticity: stay within 3 % of the main competitor and prefer a slight under‑cut when margin permits【8】.  
- Psychological price point ending in 99 is perceived as good value【9】.  

**3. Recommendation**  
Set the selling price at **₹2,599**.  
Margin = (2,599 – 1,800) ÷ 2,599 ≈ **30.8 %*

## 5. Elasticity: the same agent, opposite directions

Two products from categories that should pull the price in opposite directions.

- **Electronics, High elasticity.** A tight case: the margin floor (₹22,000 / 0.82 = ₹26,829) is only ₹161 below the competitor, and the current price is ₹1,000 above the competitor.
- **Luxury Goods, Low elasticity.** The floor (₹16,000 / 0.45 = ₹35,556) is well below the competitor's ₹38,500.

In [6]:
answer = agent.recommend(product_name="Wireless Noise-Cancelling Headphones", category="Electronics",
                         cost_price=22000, current_price=27990, target_margin=18, competitor_price=26990,
                         price_elasticity="High")
print("Knowledge retrieved:")
print(agent.sources() + "\n")
print(answer)

Knowledge retrieved:
  [1] benchmark:Electronics
  [2] margins:Electronics
  [3] trend:Electronics
  [4] decision:Sony Noise-Cancelling Headphones
  [5] decision:boAt Airdopes Wireless Earbuds
  [6] decision:25W USB-C Fast Charger
  [7] decision:LG 43-inch Smart TV
  [8] guideline:High elasticity: small price changes
  [9] guideline:Psychological price points
  [10] guideline:Never price below the margin floor

**1. Margin floor**  
Target margin = 18 % → minimum price = cost / (1 – target) = 22,000 / 0.82 = **₹26,829.27**.  
Round up to the nearest “‑99” psychological point → **₹26,899** (≥ floor)【10】.

**2. Evidence**  
- High elasticity (‑1.5 to ‑2.5) means a small price move strongly affects volume; shoppers compare online and react quickly【1】.  
- Guideline for high elasticity: change price ≤ 3‑5 % and stay within 3 % of the main competitor, preferring a slight under‑cut when margin permits【8】.  
- Competitor price is ₹26,990; our floor‑based price is 0.34 % lower, satisfying the 

In [7]:
answer = agent.recommend(product_name="Coach Leather Handbag", category="Luxury Goods",
                         cost_price=16000, target_margin=55, competitor_price=38500, price_elasticity="Low")
print("Knowledge retrieved:")
print(agent.sources() + "\n")
print(answer)

Knowledge retrieved:
  [1] benchmark:Luxury Goods
  [2] margins:Luxury Goods
  [3] trend:Luxury Goods
  [4] decision:Michael Kors Leather Handbag
  [5] decision:Ray-Ban Sunglasses
  [6] decision:Hugo Boss Perfume 100ml
  [7] decision:Tissot Automatic Watch
  [8] guideline:Low elasticity: protect price and brand
  [9] guideline:Psychological price points
  [10] guideline:Passing on cost increases

**1. Margin floor**  
Target margin = 55 % → Minimum price = Cost / (1 – 0.55) = 16,000 / 0.45 = **₹35,556**.  
Round up to a luxury‑style round figure → **₹36,000** (meets the floor and keeps a clean round price)【9】.

**2. Evidence**  
- Elasticity is low, so price changes have little impact on demand and brand perception matters【1】.  
- Historical luxury margins average 58 % (range 40‑65 %) and discounts >10 % hurt sell‑through【2】.  
- Competitors rarely discount; authorised sellers stay at or near the listed price【3】.  
- Successful past cases (Michael Kors handbag, Hugo Boss perfume) kept 

## 6. Adding a new category

What happens when the knowledge base has nothing on a category, and what changes when data arrives? The same question is asked twice about an exercise bike in Sports & Fitness: once before that category exists, and once after [`new_category.json`](new_category.json) adds 8 documents (a benchmark, margins, a competitor trend and 5 past decisions). No elasticity is given, so the agent has to find it in the knowledge base. The floor is ₹9,800 / 0.70 = ₹14,000.

In [8]:
exercise_bike = dict(product_name="Magnetic Exercise Bike", category="Sports & Fitness",
                     cost_price=9800, target_margin=30, competitor_price=14499)

print("BEFORE adding Sports & Fitness data\n")
answer = agent.recommend(**exercise_bike)
print("Knowledge retrieved:")
print(agent.sources() + "\n")
print(answer)

BEFORE adding Sports & Fitness data



Knowledge retrieved:
  [1] guideline:Categories with no history
  [2] guideline:Psychological price points
  [3] guideline:Passing on cost increases

**1. Margin floor**  
Target margin = 30 % → minimum price = cost ÷ (1 – 0.30)  
= ₹9,800 ÷ 0.70 = ₹14,000 (rounded up to meet the margin)【1】  

**2. Evidence**  
- No category benchmarks exist, so start at the floor price and stay within 5 % of the competitor (₹14,499)【1】.  
- 5 % of the competitor price = ₹724, giving an acceptable band of ₹13,774 – ₹15,224; the floor price ₹14,000 lies inside this band.  
- Psychological pricing suggests ending the price in 99 for mass‑market value perception【2】.  

**3. Recommendation**  
Set the selling price at **₹14,499**.  
Margin = (₹14,499 – ₹9,800) ÷ ₹14,499 ≈ 0.324 → 32.4 % (above the 30 % target).  
The price matches the competitor, stays within the 5 % window, and uses the “‑99” psychological cue【2】.  

**4. Risks**  
Watch sales volume and price‑sensitivity during the first 4 weeks; adjust 

In [9]:
new_documents = create_documents(load_knowledge(NEW_CATEGORY_PATH))
add_documents(store, new_documents)
print(f"Added {len(new_documents)} documents; the store now holds {store._collection.count()}.")
print(pd.DataFrame(load_knowledge(NEW_CATEGORY_PATH)["pricing_decisions"])[
    ["product", "cost", "recommended_price", "margin", "competitor_price", "elasticity", "outcome"]].to_string())

Added 8 documents; the store now holds 60.
                     product   cost  recommended_price margin  competitor_price elasticity                                                                               outcome
0       Boldfit Yoga Mat 6mm    380                699  45.6%               749     Medium                           Successful - steady impulse sales from the checkout display
1       Resistance Bands Set    280                599  53.3%               549        Low                            Successful - sold mostly in a ₹1,199 bundle with yoga mats
2      Cosco Football Size 5    420                799  47.4%               749     Medium                    Successful - school-season demand held at ₹50 above the competitor
3  Adjustable Dumbbells 20kg   2600               3999  35.0%              4299       High                                             Successful - January sales 30% above plan
4    Motorised Treadmill 2HP  24000              32999  27.3%           

In [10]:
print("AFTER adding Sports & Fitness data\n")
answer = agent.recommend(**exercise_bike)
print("Knowledge retrieved:")
print(agent.sources() + "\n")
print(answer)

AFTER adding Sports & Fitness data



Knowledge retrieved:
  [1] benchmark:Sports & Fitness
  [2] margins:Sports & Fitness
  [3] trend:Sports & Fitness
  [4] decision:Boldfit Yoga Mat 6mm
  [5] decision:Adjustable Dumbbells 20kg
  [6] decision:Resistance Bands Set
  [7] decision:Motorised Treadmill 2HP
  [8] guideline:Medium elasticity: price near the competitor
  [9] guideline:Psychological price points
  [10] guideline:Passing on cost increases

**1. Margin floor**  
Target margin = 30 % → Minimum price = Cost ÷ (1 – 0.30) = 9,800 ÷ 0.70 = **₹14,000**.  
Round up to a psychological “‑99” point → **₹14,099** (meets target margin).

**2. Evidence**  
- Big‑ticket fitness gear shows **high price elasticity** (research‑driven buying) [1].  
- Equipment historically earns 25‑35 % margin; 30 % is within the normal range [2].  
- Main competitor lists the bike at **₹14,499** [3].  
- Past treadmill case: pricing **>5 % above** competitor caused 30 % sales drop [7].  
- Guideline: stay **within 5 % of competitor** for elastic it

## Observations

Retrieval is the same on every run. The answers come from the model and vary from run to run, so the prices below are the ones in this notebook's saved outputs.

### RAG vs prompt-only: the Puma sneakers

Cost ₹1,800 and a 30% target margin give a margin floor of ₹1,800 / 0.70 = ₹2,571.43.

| | RAG agent | Prompt-only baseline |
|---|---|---|
| **Price** | ₹2,599 (30.7% margin) | ₹2,600 (30.8% margin) |
| **Margin floor** | ₹2,571.43, rounded up to the next price ending in 99 | Called ₹2,600 "the lowest price that still meets the 30% margin", although ₹2,572 already does |
| **Market position** | Entry-level branded sneakers cluster at ₹2,199-₹2,999 [3], and the Puma Smash sold 92% of its stock priced 4% below a competitor [4] | **Invented**: comparable Puma sneakers "usually sit between ₹2,500 and ₹3,000", although its rules say never to invent market data |
| **Can it be checked?** | 5 of the 10 retrieved documents are cited | No sources |

- **The two prices are almost the same; the difference is whether the reasoning can be checked.** The RAG answer's market facts come with citations to records. The baseline's market range came from nowhere.
- **The exception shows what citations are for.** The RAG answer also placed ₹2,599 "just below a typical competitor price of ~₹2,699-₹2,799". No record contains that range: the sentence's citation [8] is the guideline's "slightly below the competitor" rule, not a source for the number. With citations, a number that no cited record contains is quick to find.
- **Small arithmetic slips remain.** The RAG answer gives its margin as 30.8%; ₹799 / ₹2,599 is 30.7%.

### Elasticity tests: right evidence, not always followed

- **Electronics, High elasticity.** The floor is ₹26,829 and the competitor charges ₹26,990. The agent chose ₹26,899 (18.2% margin), ₹91 below the competitor, citing the Sony headphones that lost sales at ₹500 above a competitor [4] and the earbuds that succeeded at ₹100 below [5]. Coming down from the current ₹27,990 is a 3.9% cut, inside the guideline's 3-5% step [8].
- **Luxury Goods, Low elasticity.** The floor is ₹35,556 and the competitor charges ₹38,500. The agent rounded the floor up to ₹36,000 (55.6% margin) and recommended that, **6.5% below the competitor**. The Low elasticity guideline it cites [8] says to price at or up to 10% *above* the competitor, and its own evidence says authorised competitors rarely discount [3], yet the answer claims the price stays "within the up to 10% above competitor comfort zone". An earlier run with the same prompt and the same retrieved documents chose ₹40,000, 4% above the competitor.

Retrieval put the right rule in front of the model, and the model cited it, but citing a rule is not the same as following it. A rule that must always hold, such as "low elasticity: do not undercut the competitor", has to be checked in code. That is the job of the structured, validated output planned for a later stage.

### Adding a new category

| | Before (no Sports & Fitness data) | After (8 new documents) |
|---|---|---|
| **Retrieved** | 3 general guidelines only | Benchmark, margins, competitor trend, 4 past decisions, the Medium elasticity guideline and 2 general guidelines |
| **Price** | ₹14,499 (32.4%), equal to the competitor | ₹14,099 (30.5%), 2.8% below the competitor |
| **Reasoning** | Said no category benchmarks exist and followed the guideline for categories with no history | Big-ticket fitness gear behaves like High elasticity [1], and the treadmill priced 5% above a competitor sold 30% below plan [7] |

- **With no data, the agent did not make anything up.** It said there was no history and stayed within the no-history guideline's 5% band, though it picked the competitor's price rather than starting near the floor, as that guideline suggests.
- **With data, specific evidence moved the price.** The benchmark's note that big-ticket equipment behaves like High elasticity, and the failed treadmill, pushed the price just below the competitor. No model was retrained: adding 8 documents was enough.
- **One citation was wrong.** The answer cites the competitor trend [3] for the competitor's ₹14,499, but that number came from the product input, not the knowledge base.

### Lessons about retrieval

- **One similarity search is not enough.** It mixed categories and blurred High and Low elasticity (see *Why one similarity search is not enough*). Metadata filters fix both, and make sure every kind of evidence reaches the model.
- **A small embedding model misses links a person would see.** For the exercise bike, the treadmill is the most relevant past decision, yet it ranks only fourth of the five Sports & Fitness decisions, behind the yoga mat, the dumbbells and the resistance bands. The agent fetches 4 past decisions, not 3, partly so that the treadmill gets in. Richer documents (for example tagging decisions as "big-ticket equipment") or a stronger embedding model would make this sturdier.

### Two rules that came from testing

- An early version priced the exercise bike at ₹13,999 against a ₹14,000 floor, because it rounded down to a price ending in 99. The shared rules now say to round up, and every price in this run is at or above its floor.
- Fetching 3 past decisions left the treadmill out of the exercise-bike answer, so the agent now fetches 4.

### Limitations and next steps

- **Rules live in the prompt.** The floor, rounding and elasticity rules are instructions, not checks, so a run can break them, as the luxury answer did. Structured output with the price as a number, validated in code, would catch that.
- **Arithmetic is still the model's.** Small slips (a margin of 30.8% for 30.7%, "about 7%" for 6.5%) are easy to spot but should not happen. A calculation tool would remove them.
- **Small, hand-written knowledge base.** 31 past decisions over 7 categories show the approach, not a validated pricing model.
- **Citations are not verified.** The answer can cite the wrong record, or state a number with no citation; nothing checks this automatically.